In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
/kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___Early_blight

In [ ]:
# Write your code here
import os
import glob
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms


train_path = os.path.join(path, "PlantVillage", "train")
test_path = os.path.join(path, "PlantVillage", "test")

In [ ]:
class_labels = {
    "Early_blight": 0,
    "Late_blight": 1,
    "healthy": 2
}

In [ ]:
# Getting the images+label from the train
for folder in sorted(os.listdir(train_path)):
    prefix = folder.split("___")[1]
    if prefix == "Early_blight":
      Early_blight_images = glob.glob(f"{train_path}/{folder}/*.JPG")
      Early_blight_labels = [class_labels[prefix]] * len(Early_blight_images)

    elif prefix == "Late_blight":
      Late_blight_images = glob.glob(f"{train_path}/{folder}/*.JPG")
      Late_blight_labels = [class_labels[prefix]] * len(Late_blight_images)
    else:
      healthy_images = glob.glob(f"{train_path}/{folder}/*.JPG")
      healthy_labels = [class_labels[prefix]] * len(healthy_images)




full_train_images = Early_blight_images + Late_blight_images + healthy_images
full_train_labels = Early_blight_labels + Late_blight_labels + healthy_labels




len(full_train_images), len(full_train_labels)

In [ ]:
# Getting the images+label from the test
for folder in sorted(os.listdir(test_path)):
    prefix = folder.split("___")[1]
    if prefix == "Early_blight":
      Early_blight_images = glob.glob(f"{test_path}/{folder}/*.JPG")
      Early_blight_labels = [class_labels[prefix]] * len(Early_blight_images)

    elif prefix == "Late_blight":
      Late_blight_images = glob.glob(f"{test_path}/{folder}/*.JPG")
      Late_blight_labels = [class_labels[prefix]] * len(Late_blight_images)
    else:
      healthy_images = glob.glob(f"{test_path}/{folder}/*.JPG")
      healthy_labels = [class_labels[prefix]] * len(healthy_images)




full_test_images = Early_blight_images + Late_blight_images + healthy_images
full_test_labels = Early_blight_labels + Late_blight_labels + healthy_labels




len(full_test_images), len(full_test_labels)

In [ ]:
full_train_images[-1],full_train_labels[-1]

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class PotatoDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths  # List of image paths
        self.labels = labels  # Corresponding labels
        self.transform = transform  # Transformations to apply

    def __len__(self):
        return len(self.image_paths)  # Total number of images

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]  # Get image path
        label = self.labels[idx]  # Get corresponding label

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Apply transformations (if any)
        if self.transform:
            image = self.transform(image)

        return image, label  # Return processed image and its label

In [ ]:
# Tasks:

# Create a custom Dataset class or use ImageFolder (if applicable)
# Create the training and testing datasets, and their DataLoaders
# Use RandomRotation(15) Augmentation on the training dataset + Reize the images to 32x32
# Display some sample images with their labels


In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),  # Flip images randomly
    transforms.ToTensor()

])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()

])

train_dataset = PotatoDataset(image_paths=full_train_images, labels=full_train_labels, transform=transform)
test_dataset = PotatoDataset(image_paths=full_test_images, labels=full_test_labels, transform=test_transform)


# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
valid_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)


# Check a batch of images
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
labels_class = {v:k for k,v in class_labels.items()}
# Define mean & std for denormalization (EfficientNet Preprocessing)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [270,233,110,89,15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image & label

    # Convert tensor to numpy for visualization
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)

    # Denormalize the image
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    # Show image
    axes[i].imshow(img_np)
    name = labels_class[label]
    axes[i].set_title(f"class: {name}")
    axes[i].axis('off')

plt.show()

In [ ]:
# Write your code here
import torch.nn as nn
import torch

# Define the CNN Model
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()

        # Convolutional Layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.batch1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.batch2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.batch3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.batch4 = nn.BatchNorm2d(128)
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.batch5 = nn.BatchNorm2d(256)

        # Activation
        self.relu = nn.ReLU()

        # Pooling Layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Layers
        self.fc1 = nn.Linear(256 *1 *1, 128)
        self.fc2 = nn.Linear(128, 3)  # 3 output classes for  Potato

    def forward(self, x):
        # Convolution + ReLU + Pooling
        x = self.pool(self.relu(self.batch1(self.conv1(x))))
        x = self.pool(self.relu(self.batch2(self.conv2(x))))
        x = self.pool(self.relu(self.batch3(self.conv3(x))))
        x = self.pool(self.relu(self.batch4(self.conv4(x))))
        x = self.pool(self.relu(self.batch5(self.conv5(x))))

        # Flatten
        x = x.view(x.size(0), -1)  # (Batch, 64*4*4)

        # Fully Connected Layers
        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # Logits

        return x                # Although this is a classification problem, we didn't apply softmax. Do you know why?👀 (Hint: CrossEntropyLoss has something to do here👀)

In [ ]:
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
# Write your code here
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, valid_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
# Write your code here
# Write your code here
import torch.nn as nn
import torch

# Define the CNN Model
class CNNModel_with_residual(nn.Module):
    def __init__(self):
        super(CNNModel_with_residual, self).__init__()

        # Convolutional Layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.batch1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.batch2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.batch3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.batch4 = nn.BatchNorm2d(128)
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.batch5 = nn.BatchNorm2d(256)

        # Activation
        self.relu = nn.ReLU()

        # Pooling Layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Layers
        self.fc1 = nn.Linear(256 *1 *1, 128)
        self.fc2 = nn.Linear(128, 3)  # 3 output classes for  Potato

    def forward(self, x):
        # Convolution + ReLU + Pooling
        x1 = self.pool(self.relu(self.batch1(self.conv1(x))))
        x2 = self.pool(self.relu(self.batch2(self.conv2(x1))))
        x2 = torch.cat([x, x1], dim=0) # there is something wrong here I don't know what !!
        x3 = self.pool(self.relu(self.batch3(self.conv3(x2))))
        x3 = torch.cat([x0, x2], dim=0)
        x4 = self.pool(self.relu(self.batch4(self.conv4(x3))))
        x4 = torch.cat([x2, x3], dim=0)
        x5 = self.pool(self.relu(self.batch5(self.conv5(x4))))
        x5 = torch.cat([x3, x4], dim=0)


        # Flatten
        x = x.view(x.size(0), -1)  # (Batch, 64*4*4)

        # Fully Connected Layers
        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # Logits

        return x                # Although this is a classification problem, we didn't apply softmax. Do you know why?👀 (Hint: CrossEntropyLoss has something to do here👀)

In [ ]:
# Write your code here
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bouns = CNNModel_with_residual().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model_bouns, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model_bouns, valid_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")
